# Scene 2 – Gemini Pipeline (Clean Multi-Cell Version)

## 1. Setup & Imports

In [ ]:
# Keep your original setup cells here (installs, API keys, repo clone)

## 2. Config

In [ ]:
SCENE_NAME = "scene_2"
image_path = f"/content/PaperTheater_ai_Pipeline/data/input/{SCENE_NAME}.jpg"


## 3. Helper Functions

In [ ]:
def expand_mask(mask, kernel_size=61):
    import cv2, numpy as np
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    return cv2.dilate(mask.astype(np.uint8), kernel, 1) > 0


## 4. Control Mask Builder (ORDER BASED)

In [ ]:
def build_control_masks_for_layer(layer_context, ordered_contexts):
    import numpy as np

    order = layer_context["order"]
    max_order = max(ctx["order"] for ctx in ordered_contexts)

    visible_mask = layer_context["visible_mask"] > 0
    ownership_mask = layer_context["ownership_mask"] > 0

    H, W = visible_mask.shape
    anchor_mask = visible_mask.copy()

    prev_visible = np.zeros((H, W), dtype=bool)
    future_visible = np.zeros((H, W), dtype=bool)

    for ctx in ordered_contexts:
        vm = ctx["visible_mask"] > 0
        if ctx["order"] < order:
            prev_visible |= vm
        elif ctx["order"] > order:
            future_visible |= vm

    forbidden_mask = prev_visible.copy()

    if order == 0:
        allowed_mask = np.ones((H, W), dtype=bool)
    elif order == max_order:
        allowed_mask = expand_mask(ownership_mask.astype(np.uint8), 61)
    else:
        allowed_mask = expand_mask(ownership_mask.astype(np.uint8), 101)
        allowed_mask = np.logical_or(allowed_mask, future_visible)

    allowed_mask &= ~forbidden_mask
    allowed_mask |= anchor_mask

    return anchor_mask, allowed_mask, forbidden_mask


## 5. Focus Image Builder

In [ ]:
def apply_focus_mask_fullframe(image_rgb, anchor_mask, allowed_mask, forbidden_mask):
    import numpy as np
    beige = np.full_like(image_rgb, 235)

    out = beige.copy()
    out[~allowed_mask] = 0
    out[forbidden_mask] = 0
    out[anchor_mask] = image_rgb[anchor_mask]

    return out


## 6. Debug Visualization

In [ ]:
def show_mask_debug(image_rgb, layer_name, anchor_mask, allowed_mask, forbidden_mask, focused_input):
    import numpy as np, matplotlib.pyplot as plt

    comp = np.zeros((*anchor_mask.shape, 3), dtype=np.uint8)
    comp[allowed_mask] = [120,200,120]
    comp[forbidden_mask] = [220,80,80]
    comp[anchor_mask] = [80,140,255]

    fig, ax = plt.subplots(1,2, figsize=(10,5))
    ax[0].imshow(comp); ax[0].set_title(layer_name)
    ax[1].imshow(focused_input); ax[1].set_title("focused_input")

    for a in ax: a.axis("off")
    plt.show()


## 7. Debug Masks (RUN BEFORE GEMINI)

In [ ]:
ordered_contexts = sorted(layer_contexts, key=lambda x: x["order"])

for ctx in ordered_contexts:
    anchor, allowed, forbidden = build_control_masks_for_layer(ctx, ordered_contexts)

    focused = apply_focus_mask_fullframe(image, anchor, allowed, forbidden)

    print(ctx["layer_name"])
    show_mask_debug(image, ctx["layer_name"], anchor, allowed, forbidden, focused)
